# System Evaluation — Retrieval & Response Quality

This notebook evaluates the production retrieval and response outputs using the QA benchmark.

**Retrieval evaluation**
- Uses policy references from `qa_pairs.json`
- Reports Hit@1, Hit@3, MRR and similarity

**Response evaluation**
- Uses the already-generated response cache
- Uses BGE semantic similarity against reference answers
- Makes **no Gemini API calls**


## 1. Setup


In [1]:
from pathlib import Path
import json

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent

QA_PATH = PROJECT_ROOT / "data" / "qa_pairs.json"
CACHE_PATH = (
    PROJECT_ROOT
    / "data"
    / "evaluation_generated_responses.json"
)

print("Project root:", PROJECT_ROOT)
print("QA file:", QA_PATH.exists())
print("Response cache:", CACHE_PATH.exists())


Project root: E:\HCL Guvi\Banking Support & Fraud\Banking Support & Fraud Intelligence System
QA file: True
Response cache: True


## 2. Retrieval Evaluation


In [2]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.evaluator import (
    load_qa_pairs,
    evaluate_retrieval,
)
from src.rag.retriever import retrieve_context

qa_df = load_qa_pairs()

def normalize_intent(category):
    return {
        "Fraud": "Fraud/Unauthorized",
        "Fraud/Unauthorized": "Fraud/Unauthorized",
        "Loan": "Loan",
        "KYC": "KYC",
        "Account Access": "Account Access",
    }.get(category, category)


def policy_retriever_for_eval(
    query,
    intent,
    top_k=3,
):
    context = retrieve_context(
        query=query,
        intent=normalize_intent(intent),
        policy_top_k=top_k,
        history_top_k=0,
    )
    return context["policy_results"]


retrieval_results, retrieval_metrics = (
    evaluate_retrieval(
        retriever_function=policy_retriever_for_eval,
        qa_df=qa_df,
        top_k=3,
    )
)

print("Retrieval metrics:")
for metric, value in retrieval_metrics.items():
    print(f"{metric:<35}: {value:.4f}")


e:\HCL Guvi\Banking Support & Fraud\Banking Support & Fraud Intelligence System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5792.63it/s]


Retrieval metrics:
Hit@1                              : 0.9412
Hit@3                              : 1.0000
MRR                                : 0.9706
Mean Top-1 Similarity              : 0.6984
Mean Relevant Similarity           : 0.6967


In [3]:
display(
    retrieval_results[
        [
            "id",
            "category",
            "expected_policy",
            "rank",
            "hit_at_1",
            "hit_at_3",
            "reciprocal_rank",
            "top1_similarity",
            "relevant_similarity",
        ]
    ]
)


,id,category,expected_policy,rank,hit_at_1,hit_at_3,reciprocal_rank,top1_similarity,relevant_similarity
0,QA001,Fraud,fraud_handling_policy.txt,1,1,1,1.0,0.6977,0.6977
1,QA002,Fraud,fraud_handling_policy.txt,1,1,1,1.0,0.7452,0.7452
2,QA003,Fraud,fraud_handling_policy.txt,2,0,1,0.5,0.7005,0.6724
3,QA004,Fraud,fraud_handling_policy.txt,1,1,1,1.0,0.6409,0.6409
4,QA005,Fraud,fraud_handling_policy.txt,1,1,1,1.0,0.6724,0.6724
5,QA006,Loan,loan_processing_policy.txt,1,1,1,1.0,0.8873,0.8873
6,QA007,Loan,loan_processing_policy.txt,1,1,1,1.0,0.7168,0.7168
7,QA008,Loan,loan_processing_policy.txt,1,1,1,1.0,0.7221,0.7221
8,QA009,Loan,loan_processing_policy.txt,1,1,1,1.0,0.7083,0.7083
9,QA010,Loan,loan_processing_policy.txt,1,1,1,1.0,0.6201,0.6201


## 3. Load Cached Gemini Responses


In [4]:
with open(
    CACHE_PATH,
    "r",
    encoding="utf-8",
) as file:
    generated_cache = json.load(file)

print(
    "Cached responses:",
    len(generated_cache),
)


Cached responses: 20


## 4. Response Semantic Evaluation


In [5]:
from src.evaluation.evaluator import (
    evaluate_response_similarity,
)

evaluation_rows = []

for _, row in qa_df.iterrows():

    qa_id = str(row["id"])

    if qa_id not in generated_cache:
        continue

    evaluation_rows.append(
        {
            "id": qa_id,
            "category": row["category"],
            "question": row["question"],
            "reference_answer": row["answer"],
            "generated_answer": generated_cache[
                qa_id
            ]["generated_answer"],
        }
    )

response_eval_df = pd.DataFrame(
    evaluation_rows
)

response_results, response_metrics = (
    evaluate_response_similarity(
        questions=response_eval_df["question"].tolist(),
        reference_answers=response_eval_df[
            "reference_answer"
        ].tolist(),
        generated_answers=response_eval_df[
            "generated_answer"
        ].tolist(),
    )
)

response_results.insert(
    0,
    "id",
    response_eval_df["id"].values,
)

response_results.insert(
    1,
    "category",
    response_eval_df["category"].values,
)

print("Response metrics:")
for metric, value in response_metrics.items():
    print(f"{metric:<35}: {value:.4f}")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4493.20it/s]


Response metrics:
Mean Similarity                    : 0.8135
Median Similarity                  : 0.8377
Minimum Similarity                 : 0.6100
Maximum Similarity                 : 0.9468
Similarity >= 0.75                 : 0.6500


## 5. Response Performance by Category


In [6]:
category_metrics = (
    response_results
    .groupby("category")["semantic_similarity"]
    .agg(
        [
            "count",
            "mean",
            "median",
            "min",
            "max",
        ]
    )
    .reset_index()
)

display(
    category_metrics.round(4)
)


,category,count,mean,median,min,max
0,Account Access,3,0.9086,0.9065,0.8726,0.9468
1,Fraud,6,0.7609,0.7522,0.6830,0.8453
2,KYC,5,0.7562,0.7280,0.6100,0.9187
3,Loan,6,0.8663,0.8805,0.7491,0.9294


## 6. Lowest-Scoring Responses


In [7]:
weakest_responses = (
    response_results
    .sort_values(
        "semantic_similarity"
    )
    [
        [
            "id",
            "category",
            "question",
            "semantic_similarity",
        ]
    ]
    .head(5)
)

display(
    weakest_responses.round(
        {"semantic_similarity": 4}
    )
)


,id,category,question,semantic_similarity
11,QA012,KYC,My Aadhaar name and bank name don't match. How...,0.6100
2,QA003,Fraud,Someone used my OTP to transfer money. Can I g...,0.6830
16,QA017,Fraud,I got a call from someone claiming to be from ...,0.6899
3,QA004,Fraud,Multiple small transactions are showing up tha...,0.7078
10,QA011,KYC,Why is my account restricted? I submitted KYC ...,0.7185


## 7. Policy-Covered vs All Responses


In [8]:
policy_covered_ids = set(
    retrieval_results["id"].astype(str)
)

response_results["policy_covered"] = (
    response_results["id"].astype(str)
    .isin(policy_covered_ids)
)

coverage_summary = (
    response_results
    .groupby("policy_covered")[
        "semantic_similarity"
    ]
    .agg(
        [
            "count",
            "mean",
            "median",
        ]
    )
    .rename(
        index={
            True: "Policy-covered",
            False: "Not policy-covered",
        }
    )
)

display(
    coverage_summary.round(4)
)


,count,mean,median
policy_covered,,,
Not policy-covered,3,0.9086,0.9065
Policy-covered,17,0.7967,0.8061


## 8. Save Evaluation Artifacts

These files are consumed by the Streamlit Evaluation tab.


In [9]:
retrieval_results.to_csv(
    PROJECT_ROOT
    / "data"
    / "retrieval_evaluation_results.csv",
    index=False,
    encoding="utf-8",
)

response_results.to_csv(
    PROJECT_ROOT
    / "data"
    / "response_evaluation_results.csv",
    index=False,
    encoding="utf-8",
)

print("Evaluation files saved.")


Evaluation files saved.


## Evaluation Notes

- Retrieval Hit@k and MRR use the policy reference in the QA benchmark.
- Response semantic similarity measures semantic alignment with the reference answer.
- Semantic similarity is **not the same as factual accuracy**.
- A low response score should be inspected against the retrieved authoritative policy before changing the system.
